In [3]:
import urllib.request

# "No Write In" version (9 candidates, cleaner)
url = "https://raw.githubusercontent.com/PrefLib/PrefLib-Data/main/datasets/00018%20-%20minneapolis/00018-00000002.soi"
response = urllib.request.urlopen(url)
data = response.read().decode("utf-8")

In [4]:
data

'# FILE NAME: 00018-00000002.soi\n# TITLE: 2009 Minneapolis Park and Recreation Commissioner At-Large Election - No Write In\n# DESCRIPTION: \n# DATA TYPE: soi\n# MODIFICATION TYPE: original\n# RELATES TO: \n# RELATED FILES: 00018-00000002.toc\n# PUBLICATION DATE: 2014-07-09\n# MODIFICATION DATE: 2022-09-16\n# NUMBER ALTERNATIVES: 9\n# NUMBER VOTERS: 36655\n# NUMBER UNIQUE ORDERS: 470\n# ALTERNATIVE NAME 1: "Nancy Bernard"\n# ALTERNATIVE NAME 2: "John Butler"\n# ALTERNATIVE NAME 3: "John Erwin"\n# ALTERNATIVE NAME 4: "Bob Fine"\n# ALTERNATIVE NAME 5: "Mary Merrill Anderson"\n# ALTERNATIVE NAME 6: "Tom Nordyke"\n# ALTERNATIVE NAME 7: "David Wahstedt"\n# ALTERNATIVE NAME 8: "Annie Young"\n# ALTERNATIVE NAME 9: "Write In"\n3761: 4\n2065: 8\n1570: 7\n1484: 5\n1104: 3\n1095: 3,8,6\n947: 6\n735: 1\n615: 3,5,6\n593: 5,3,6\n524: 3,6,8\n516: 8,1,5\n485: 8,3,6\n448: 2\n439: 3,6,5\n389: 3,8\n339: 5,8,1\n327: 8,3\n324: 8,5,1\n318: 8,3,5\n297: 5,6,3\n287: 4,8\n284: 4,5\n277: 1,5,8\n260: 6,3,5\n259:

In [5]:
import pandas as pd

url = "https://vote.minneapolismn.gov/media/-www-content-assets/documents/2021-Mayor-Cast-Vote-Record.csv"
df = pd.read_csv(url)
print(df.head())
print(df.columns.tolist())

                Precinct   1st Choice            2nd Choice  3rd Choice  Count
0  MINNEAPOLIS W-12 P-13   Jacob Frey             undervote   undervote      1
1  MINNEAPOLIS W-12 P-13  Mark Globus           Mike Winter  Jacob Frey      1
2  MINNEAPOLIS W-12 P-13   Jacob Frey  Kevin "No Body" Ward   undervote      1
3  MINNEAPOLIS W-12 P-13   Jacob Frey             undervote   undervote      1
4  MINNEAPOLIS W-12 P-13   Jacob Frey            Kate Knuth     AJ Awed      1
['Precinct', '1st Choice', '2nd Choice', '3rd Choice', 'Count']


In [13]:
import numpy as np
from collections import Counter

# ── Collect all unique candidates across all choice columns ───────────────────
SKIP = {"undervote", "overvote", "UWI"}  # undervotes, overvotes, unresolved write-ins

all_names = set()
for col in ["1st Choice", "2nd Choice", "3rd Choice"]:
    all_names.update(df[col].dropna().unique())
all_names -= SKIP

# Sort for reproducibility
candidates = sorted(all_names)
cand_to_idx = {c: i for i, c in enumerate(candidates)}
n_cand = len(candidates)

print(f"Candidates ({n_cand}):")
for i, c in enumerate(candidates):
    print(f"  {i:2d}. {c}")

# ── Count 1st-choice votes per candidate ──────────────────────────────────────
first_counts = df["1st Choice"].value_counts()
print(f"\n1st-choice vote totals (top 10):")
for name, cnt in first_counts.head(10).items():
    pct = 100 * cnt / len(df)
    print(f"  {name:<30s}  {cnt:>6,d}  ({pct:.1f}%)")

Candidates (18):
   0. AJ Awed
   1. Bob "Again" Carney Jr
   2. Christopher Robin "CRZ" Zimmerman
   3. Christopher W David
   4. Clint Conner
   5. Doug Nelson
   6. Jacob Frey
   7. Kate Knuth
   8. Kevin "No Body" Ward
   9. Laverne Turner
  10. Marcus Harcus
  11. Mark Globus
  12. Mike Winter
  13. Nate "Honey Badger" Atkins
  14. Paul E. Johnson
  15. Perry, Jerrell
  16. Sheila Nezhad
  17. Troy Benjegerdes

1st-choice vote totals (top 10):
  Jacob Frey                      61,469  (42.3%)
  Sheila Nezhad                   30,335  (20.9%)
  Kate Knuth                      26,444  (18.2%)
  AJ Awed                          6,823  (4.7%)
  Laverne Turner                   4,604  (3.2%)
  Clint Conner                     4,290  (3.0%)
  Bob "Again" Carney Jr            2,778  (1.9%)
  undervote                        1,547  (1.1%)
  Marcus Harcus                    1,183  (0.8%)
  Nate "Honey Badger" Atkins       1,176  (0.8%)


In [14]:
# ── Convert CVR to partial strict rankings ────────────────────────────────────
# Each row is one ballot.  We take the ranked candidates in order and produce
# a partial ranking (length 1-3) over the 0-indexed candidate set.
# Unranked candidates are simply absent -> partial ranking for the Mallows model.

rng_cvr = np.random.default_rng(42)
rankings_list: list[list[int]] = []

choice_cols = ["1st Choice", "2nd Choice", "3rd Choice"]

for _, row in df.iterrows():
    ballot: list[int] = []
    seen = set()
    for col in choice_cols:
        name = row[col]
        if isinstance(name, str) and name not in SKIP and name in cand_to_idx:
            idx = cand_to_idx[name]
            if idx not in seen:            # skip duplicates
                ballot.append(idx)
                seen.add(idx)
    if len(ballot) >= 1:                   # keep only non-empty ballots
        rankings_list.append(ballot)

N = len(rankings_list)
lengths = np.array([len(r) for r in rankings_list])

print(f"Ballots retained: {N:,} / {len(df):,}")
print(f"Ballot lengths: min={lengths.min()}, mean={lengths.mean():.2f}, max={lengths.max()}")
print(f"Missingness: mean={(1 - lengths/n_cand).mean():.1%}")

# ── Subsample for tractability  ───────────────────────────────────────────────
# 145k voters x 18 candidates is large; subsample to keep MCMC reasonable.
MAX_VOTERS = 5000
if N > MAX_VOTERS:
    idx = rng_cvr.choice(N, size=MAX_VOTERS, replace=False)
    rankings_sub = [rankings_list[i] for i in idx]
    print(f"\nSubsampled to {MAX_VOTERS:,} voters for MCMC.")
else:
    rankings_sub = rankings_list
    print(f"\nUsing all {N:,} voters.")

N_sub = len(rankings_sub)
print(f"Rankings: {N_sub}, Items: {n_cand}")

Ballots retained: 143,890 / 145,337
Ballot lengths: min=1, mean=2.19, max=3
Missingness: mean=87.8%

Subsampled to 5,000 voters for MCMC.
Rankings: 5000, Items: 18


In [16]:
# ── Spectral clustering initialization ─────────────────────────────────────────
# init_spectral_with_z requires equal-length rankings.  With partial ballots
# (length 1-3 out of 18 candidates) we use an agreement-based spectral
# clustering on the common-item subset, then build paired-block consensus.

from model import ClusterParams, MixtureRankingModel
from model.initialization import init_spectral_with_z
import random as _random
from sklearn.cluster import SpectralClustering
from collections import defaultdict

C = 20          # initial number of clusters
gamma = 1.0     # PY concentration
delta = 0.5     # PY discount

# ── 1. Build pairwise agreement matrix over ballots ──────────────────────────
# For partial rankings: agree on (a, b) if both voters rank a and b in the same
# relative order.  Normalise by the number of candidate pairs both voters ranked.

def partial_agreement(r1, r2):
    """Fraction of shared pairs ranked in the same order."""
    s1 = set(r1)
    s2 = set(r2)
    common = s1 & s2
    if len(common) < 2:
        return 0.0
    # Build position maps
    pos1 = {item: i for i, item in enumerate(r1) if item in common}
    pos2 = {item: i for i, item in enumerate(r2) if item in common}
    items = sorted(common)
    agree = 0
    total = 0
    for a_idx in range(len(items)):
        for b_idx in range(a_idx + 1, len(items)):
            a, b = items[a_idx], items[b_idx]
            if a in pos1 and b in pos1 and a in pos2 and b in pos2:
                total += 1
                if (pos1[a] - pos1[b]) * (pos2[a] - pos2[b]) > 0:
                    agree += 1
    return agree / total if total > 0 else 0.0

# For 5000 voters, full N^2 is too expensive. Use a random subsample for
# spectral clustering, then assign remaining voters to nearest cluster.
N_spectral = min(N_sub, 2000)
rng_sp = np.random.default_rng(42)
sp_idx = rng_sp.choice(N_sub, size=N_spectral, replace=False)
sp_rankings = [rankings_sub[i] for i in sp_idx]

print(f"Building agreement matrix for {N_spectral} voters...")
import time as _time
t0 = _time.time()

# Use first-choice based affinity (fast, interpretable for RCV data)
# Build a feature matrix: one-hot of 1st, 2nd, 3rd choice
from scipy.sparse import lil_matrix
feat = lil_matrix((N_spectral, n_cand * 3), dtype=np.float32)
for i, r in enumerate(sp_rankings):
    for rank_pos, item in enumerate(r[:3]):
        feat[i, rank_pos * n_cand + item] = 1.0
feat_csr = feat.tocsr()

# Cosine similarity -> affinity
from sklearn.metrics.pairwise import cosine_similarity
affinity = cosine_similarity(feat_csr)
affinity = np.clip(affinity, 0, None)  # ensure non-negative
np.fill_diagonal(affinity, 0)

print(f"  Affinity matrix: {_time.time() - t0:.1f}s")

# ── 2. Spectral clustering ──────────────────────────────────────────────────
sc = SpectralClustering(
    n_clusters=C,
    affinity="precomputed",
    random_state=42,
    n_init=10,
)
sp_labels = sc.fit_predict(affinity)
print(f"  Spectral clustering done. Sizes: {np.bincount(sp_labels, minlength=C).tolist()}")

# ── 3. Assign ALL voters to clusters (nearest centroid in feature space) ─────
# Build features for all voters
feat_all = lil_matrix((N_sub, n_cand * 3), dtype=np.float32)
for i, r in enumerate(rankings_sub):
    for rank_pos, item in enumerate(r[:3]):
        feat_all[i, rank_pos * n_cand + item] = 1.0
feat_all_csr = feat_all.tocsr()

# Compute cluster centroids from spectral subset
centroids = np.zeros((C, n_cand * 3), dtype=np.float32)
for c_i in range(C):
    mask = sp_labels == c_i
    if mask.sum() > 0:
        centroids[c_i] = feat_csr[mask].mean(axis=0).A1

# Assign all voters to nearest centroid
from sklearn.metrics.pairwise import cosine_distances
dists = cosine_distances(feat_all_csr, centroids)
z_init = list(dists.argmin(axis=1))

print(f"  Full assignment: {np.bincount(z_init, minlength=C).tolist()}")

# ── 4. Build initial cluster params with paired-block structure ──────────────
_blocks = [[i, i + 1] for i in range(0, n_cand - 1, 2)]
if n_cand % 2 == 1:
    _blocks.append([n_cand - 1])

clusters = [
    ClusterParams(
        blocks=[b[:] for b in _blocks],
        theta=1.0,
        gamma=gamma,
        delta=delta,
    )
    for _ in range(C)
]

init_mu = [1 / C] * C

print(f"\nInitialised {C} clusters, {n_cand} items, {N_sub} voters")
print(f"Blocks per cluster: {len(_blocks)} (consecutive pairs, MCMC will adjust)")
t_total = _time.time() - t0
print(f"Total init time: {t_total:.1f}s")

Building agreement matrix for 2000 voters...
  Affinity matrix: 0.1s
  Spectral clustering done. Sizes: [1209, 4, 5, 4, 7, 2, 2, 32, 21, 3, 23, 1, 53, 25, 62, 54, 5, 415, 60, 13]
  Full assignment: [2221, 23, 12, 23, 38, 27, 16, 99, 57, 11, 79, 10, 569, 186, 191, 126, 17, 963, 267, 65]

Initialised 20 clusters, 18 items, 5000 voters
Blocks per cluster: 9 (consecutive pairs, MCMC will adjust)
Total init time: 9.5s


In [18]:
# ── Estimate tie-penalty p_hat ────────────────────────────────────────────────
# estimate_p_multicluster requires equal-length rankings.  Use only the
# voters who ranked exactly 3 candidates (the maximum) as a representative
# sample for estimation.

from helper_functions.Estimate_p import estimate_p_multicluster

# Find ballots with max length (3) that are in our subsample
full_mask = [i for i, r in enumerate(rankings_sub) if len(r) == 3]
print(f"Estimating p_hat from {len(full_mask)} ballots with 3 ranked candidates")

if len(full_mask) >= 50:
    sub_rl = [rankings_sub[i] for i in full_mask]
    sub_z  = [z_init[i] for i in full_mask]
    p_est = estimate_p_multicluster(
        rankings=sub_rl,
        cluster_assignments=sub_z,
        delta=delta,
        lam=1.0,
        n_mc_pi1=5000,
        grid_size=1000,
    )
    p_hat = p_est["p_hat_global"]
    print(f"Estimated p_hat: {p_hat:.4f}")
else:
    p_hat = 0.5
    print(f"Too few full-length ballots, using default p_hat = {p_hat}")

print(f"\nFinal tie-penalty for MCMC: p = {p_hat:.4f}")

Estimating p_hat from 2322 ballots with 3 ranked candidates
Estimated p_hat: 0.3503

Final tie-penalty for MCMC: p = 0.3503


In [20]:
# ── Run MCMC ──────────────────────────────────────────────────────────────────
model = MixtureRankingModel(
    rankings=rankings_sub,
    n_items=n_cand,
    init_clusters=clusters,
    init_z=z_init,
    init_mu=init_mu,
    seed=42,
    verbose=True,
    init_theta=1.0,
)

n_iter = 15000
burn_in = 10000
thin = 1

final_state, samples = model.run_mcmc(
    n_iter=n_iter,
    burn_in=burn_in,
    thin=thin,
    save_samples=True,
    save_tau=True,
    save_theta=True,
    n_item_moves_per_cluster=1,
    gamma=gamma,
    delta=delta,
    a_theta=4,
    b_theta=1,
    theta_jump=10,
    ranking_jump=5,
    use_annealing=True,
    temp_min=0.1,
    temp_max=1.0,
    tie_penalty=p_hat,
)

model.print_acceptance_summary()

[Model] Initialized: N=5000 assessors, n=18 items, C=20 clusters, n_pairs=153
[Model] Compute:  CPU  (GPU unavailable — torch not installed)
[Model] U_all:    5000×153  (2.9 MB float32, on CPU)
[Model] Parallel: N threshold = disabled
[Model] Partial rankings: 5000/5000 assessors have missing items
[Model]   Missing items per assessor: min=15, max=17, mean=15.8
  Cluster 0: 13 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 1: 12 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 2: 10 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 3: 10 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 4: 11 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 5: 14 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 6: 11 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 7: 11 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 8: 13 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 9: 11 blocks, theta=1.000, gamma=1.000, delta=0.500
  Cluster 10: 11

KeyboardInterrupt: 